<a href="https://colab.research.google.com/github/ajasjaleel/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajasjaleel/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row represents one pseudonymized client, content item, and report date in the daily performance table. The available history is unbalanced across clients, so the usable time window depends on each client's observed data coverage.


In [57]:
# Verify row grain and available date window

print("Daily performance table:")
print(con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""").df())

Daily performance table:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   row_count  clients  content_items   min_date   max_date
0   78835655       70         427292 2025-01-27 2026-06-30


## 2. Fields: feature / label / context / excluded

Features are fields available before or during the prediction decision, such as historical performance and content attributes. The label is the outcome we want to measure or predict. Context fields describe the observation, such as client, content, and date. Fields that could leak future information or identify a client are excluded.

In [58]:
# Inspect available columns and classify the fields used in the contract

schema = con.execute(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""").df()

display(schema)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)


The queries below verify the stated grain, row counts, date window, and missingness. These checks are measured from the available warehouse rather than assumed from the table description.

In [59]:
# Basic verification checks

daily = f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')"

checks = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT client_hash_id) AS client_count,
        COUNT(DISTINCT content_hash_id) AS content_count,
        COUNT(DISTINCT report_date) AS date_count,
        MIN(report_date) AS min_report_date,
        MAX(report_date) AS max_report_date,
        COUNT(*) - COUNT(client_hash_id) AS missing_client_id,
        COUNT(*) - COUNT(content_hash_id) AS missing_content_id,
        COUNT(*) - COUNT(report_date) AS missing_report_date
    FROM {daily}
""").df()

display(checks)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,client_count,content_count,date_count,min_report_date,max_report_date,missing_client_id,missing_content_id,missing_report_date
0,78835655,70,427292,520,2025-01-27,2026-06-30,0,0,0


In [60]:
# Check whether the claimed grain has duplicate rows

duplicate_check = con.execute(f"""
    SELECT
        COUNT(*) AS duplicate_groups
    FROM (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            COUNT(*) AS n
        FROM {daily}
        GROUP BY
            client_hash_id,
            content_hash_id,
            report_date
        HAVING COUNT(*) > 1
    )
""").df()

display(duplicate_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_groups
0,6390


## 4. Data limits

This data cannot provide a balanced history for every client because clients have different data-start dates. Some observations are available only from GSC or another source, so missingness can reflect access rather than absence of activity. Fixed 90-day query windows can also overlap reporting periods, so they should not be treated as independent observations. The dataset supports directional and decision-support analysis, not causal conclusions.

In [61]:
from datasets import load_dataset
import pandas as pd

# Load the client dimension table
clients_ds = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_clients",
    split="train"
)

clients_df = clients_ds.to_pandas()

# Convert date fields to real datetime values
clients_df["gsc_data_start"] = pd.to_datetime(
    clients_df["gsc_data_start"],
    errors="coerce"
)

clients_df["ga4_data_start"] = pd.to_datetime(
    clients_df["ga4_data_start"],
    errors="coerce"
)

# Check the observed client data-start dates
start_dates = pd.DataFrame({
    "earliest_gsc_start": [clients_df["gsc_data_start"].min()],
    "latest_gsc_start": [clients_df["gsc_data_start"].max()],
    "earliest_ga4_start": [clients_df["ga4_data_start"].min()],
    "latest_ga4_start": [clients_df["ga4_data_start"].max()]
})

display(start_dates)

,earliest_gsc_start,latest_gsc_start,earliest_ga4_start,latest_ga4_start
0,2025-01-27,2026-06-02,2025-10-29,2026-06-01


In [62]:
# Check the distribution of source access profiles

access_profile = (
    clients_df["access_profile"]
    .value_counts(dropna=False)
    .rename_axis("access_profile")
    .reset_index(name="client_count")
)

display(access_profile)

,access_profile,client_count
0,gsc_and_ga4,53
1,no_search_or_analytics_access,26
2,gsc_only,14
3,source_only_missing_client_dimension,10
4,ga4_only,1


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.